# Solution 02 — basic consumer

In [ ]:
from confluent_kafka import Consumer

BROKER = 'redpanda:29092'

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'live-map',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['rider-locations', 'orders'])
print('Subscribed.')

## Step 3 — read existing messages

In [ ]:
messages_read, empty_polls = 0, 0
while messages_read < 20 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    key   = msg.key().decode()   if msg.key()   else 'None'
    value = msg.value().decode() if msg.value() else 'None'
    print(f'[{messages_read}] {msg.topic()} P{msg.partition()} '
          f'offset={msg.offset()} key={key}')
    print(f'     {value}')

## Task B — fresh group reads from the start

Notice this consumer reads the *same* messages again, despite the first one having read them already. The new `group.id` has no stored offset, so `auto.offset.reset='earliest'` kicks in.

The first `poll()` calls may return `None` while the group is being set up, so we poll a few more times than we want messages.

In [ ]:
consumer.close()
consumer2 = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'analytics',
    'auto.offset.reset': 'earliest',
})
consumer2.subscribe(['rider-locations', 'orders'])

read, empty_polls = 0, 0
while read < 10 and empty_polls < 5:
    msg = consumer2.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; read += 1
    print(f'[{read}] {msg.topic()} P{msg.partition()} '
          f'offset={msg.offset()}')
consumer2.close()